In [2]:
import pandas as pd 

In [3]:
orders = pd.read_csv('orders.csv')

In [4]:
orders.head()

,order_id,order_date,customer_id,zip,order_status,payment_method,device_type,order_source
0,1,2012-07-04,58578,1109,delivered,credit_card,desktop,paid_search
1,2,2012-07-04,58621,1330,returned,cod,mobile,paid_search
2,3,2012-07-04,58811,1473,delivered,credit_card,desktop,direct
3,4,2012-07-04,59453,2360,delivered,credit_card,desktop,referral
4,6,2012-07-06,57821,2886,delivered,paypal,mobile,email_campaign


In [5]:
customers = pd.read_csv('customers.csv')
customers.head()

,customer_id,zip,city,signup_date,gender,age_group,acquisition_channel
0,1,15201,Hai Phong,2021-12-30,Female,35-44,social_media
1,2,15201,Hai Phong,2013-12-27,Female,45-54,email_campaign
2,3,15201,Hai Phong,2018-07-24,Female,18-24,organic_search
3,4,15201,Hai Phong,2017-11-29,Male,35-44,referral
4,5,15201,Hai Phong,2022-09-23,Male,55+,organic_search


In [6]:
data = pd.merge(orders, customers, on='customer_id', how='left')

In [7]:
data

,order_id,order_date,customer_id,zip_x,order_status,payment_method,device_type,order_source,zip_y,city,signup_date,gender,age_group,acquisition_channel
0,1,2012-07-04,58578,1109,delivered,credit_card,desktop,paid_search,1109,Hanoi,2020-06-06,Female,35-44,social_media
1,2,2012-07-04,58621,1330,returned,cod,mobile,paid_search,1330,Phu Ly,2021-11-03,Female,18-24,social_media
2,3,2012-07-04,58811,1473,delivered,credit_card,desktop,direct,1473,Lao Cai,2020-09-18,Female,35-44,direct
3,4,2012-07-04,59453,2360,delivered,credit_card,desktop,referral,2360,Son Tay,2016-05-29,Male,45-54,direct
4,6,2012-07-06,57821,2886,delivered,paypal,mobile,email_campaign,2886,Uong Bi,2017-07-11,Male,18-24,social_media
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
646940,834372,2022-12-31,19490,33907,delivered,credit_card,mobile,email_campaign,33907,Bac Ninh,2018-10-16,Male,18-24,paid_search
646941,834377,2022-12-31,73046,37091,delivered,credit_card,mobile,referral,37091,Ha Long,2022-11-12,Male,45-54,organic_search
646942,834387,2022-12-31,107723,80516,delivered,credit_card,mobile,email_campaign,80516,Kon Tum,2022-01-02,Male,45-54,organic_search
646943,834392,2022-12-31,139431,93510,delivered,paypal,desktop,direct,93510,Ben Tre,2021-06-12,Male,25-34,referral


In [8]:
data['count_order'] = data.groupby('customer_id')['order_id'].transform('count')

In [9]:
df = data[data['count_order'] > 1]

In [10]:
df['order_date'] = pd.to_datetime(df['order_date'], errors='coerce')


C:\Users\Admin\AppData\Local\Temp\ipykernel_28380\4257316843.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['order_date'] = pd.to_datetime(df['order_date'], errors='coerce')


In [11]:
df = df.sort_values(["customer_id", "order_date"])


In [12]:
df['gap_days'] = df.groupby('customer_id')['order_date'].diff().dt.days

In [13]:
df['gap_days'].median()

144.0

q2: Phân khúc sản phẩm (segment) nào trong products.csv có tỷ suất lợi nhuận gộp
trung bình cao nhất, với công thức (price − cogs)/price?

In [14]:
products = pd.read_csv('products.csv')

In [15]:
products.head()

,product_id,product_name,category,segment,size,color,price,cogs
0,536,SaigonFlex UC-01,Streetwear,Everyday,S,green,11059.650000,9704.842875
1,537,SaigonFlex UC-02,Streetwear,Everyday,M,silver,9523.076013,5393.870254
2,538,SaigonFlex UC-03,Streetwear,Everyday,L,pink,15951.633158,11371.919278
3,539,SaigonFlex UC-04,Streetwear,Everyday,XL,yellow,15753.717299,8573.172954
4,540,SaigonFlex UC-05,Streetwear,Everyday,S,red,15766.334536,14063.570406


In [16]:
products['profit'] = (products['price'] - products['cogs']) / products['price']

In [17]:
products['segment'].value_counts()

segment
Activewear     598
Everyday       405
Performance    347
Balanced       306
Standard       262
Premium        177
All-weather    169
Trendy         148
Name: count, dtype: int64

In [18]:
segment = products.groupby('segment')['profit'].mean().sort_values(ascending=False)

In [19]:
segment.head(1)

segment
Standard    0.313442
Name: profit, dtype: float64

Q3. Trong các bản ghi trả hàng liên kết với sản phẩm thuộc danh mục Streetwear (join
returns với products theo product_id), lý do trả hàng nào xuất hiện nhiều nhất?

In [20]:
returns = pd.read_csv('returns.csv')
products = pd.read_csv('products.csv')

In [21]:
q3 = pd.merge(returns, products, on='product_id', how='left')

In [22]:
q3.head()

,return_id,order_id,product_id,return_date,return_reason,return_quantity,refund_amount,product_name,category,segment,size,color,price,cogs
0,RET-000001,2,609,2012-07-25,late_delivery,6,52458.01,SaigonFlex UC-74,Streetwear,Everyday,M,yellow,10426.571034,8987.704231
1,RET-000002,32,1862,2012-07-16,wrong_size,2,5141.37,SaigonCore YY-57,GenZ,Trendy,L,orange,2656.232069,1842.628186
2,RET-000003,35,2359,2012-07-16,wrong_size,1,5315.95,VietMotion UC-07,Streetwear,Everyday,XL,yellow,5399.825901,3136.758866
3,RET-000004,47,1449,2012-07-11,wrong_size,4,6493.75,VietMode RP-41,Outdoor,Activewear,M,yellow,1802.115000,1575.769356
4,RET-000005,47,1450,2012-07-25,wrong_size,1,1740.76,VietMode RP-42,Outdoor,Activewear,L,red,1802.115000,1309.056336


In [23]:
data3 = q3[q3['category'] == 'Streetwear']

In [24]:
data3['return_reason'].value_counts()

return_reason
wrong_size          7626
defective           4330
not_as_described    3854
changed_mind        3830
late_delivery       2159
Name: count, dtype: int64

Q4. Trong web_traffic.csv, nguồn truy cập (traffic_source) nào có tỷ lệ thoát trung
bình (bounce_rate) thấp nhất trên tất cả các ngày xuất hiện nguồn đó trong cột traffic_source?

In [25]:
traffic = pd.read_csv('web_traffic.csv')

In [26]:
traffic.head()

,date,sessions,unique_visitors,page_views,bounce_rate,avg_session_duration_sec,traffic_source
0,2013-01-01,9760,7253,39093,0.00514,102.9,organic_search
1,2013-01-02,10456,8151,47611,0.00406,120.5,organic_search
2,2013-01-03,10076,7458,36963,0.00401,263.6,direct
3,2013-01-04,9973,8063,53078,0.00562,151.8,direct
4,2013-01-05,10223,7882,36790,0.00525,168.6,referral


In [27]:
temp=traffic['traffic_source'].value_counts()

In [28]:
temp = pd.DataFrame(temp)

In [29]:
tmp=pd.DataFrame(traffic.groupby(['traffic_source'])['bounce_rate'].mean().sort_values(ascending=True))

In [30]:
tmp[tmp['bounce_rate'] == tmp['bounce_rate'].min()].index

Index(['email_campaign'], dtype='object', name='traffic_source')

Q5. Tỷ lệ phần trăm các dòng trong order_items.csv có áp dụng khuyến mãi (tức là promo_id
không null) xấp xỉ là bao nhiêu?

In [31]:
order_items = pd.read_csv('order_items.csv')
order_items.head()

C:\Users\Admin\AppData\Local\Temp\ipykernel_28380\3114237517.py:1: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  order_items = pd.read_csv('order_items.csv')


,order_id,product_id,quantity,unit_price,discount_amount,promo_id,promo_id_2
0,1,2400,7,1138.22,0.0,NaN,NaN
1,2,609,7,10166.25,0.0,NaN,NaN
2,3,396,3,11220.33,0.0,NaN,NaN
3,4,635,5,10639.25,0.0,NaN,NaN
4,6,1935,1,1597.84,0.0,NaN,NaN


In [32]:
order_items.shape[0]

714669

In [33]:
order_items[order_items['discount_amount']!=0].shape[0]

276316

In [34]:
percentage = (order_items[order_items['discount_amount']!=0].shape[0] / order_items.shape[0])*100

In [35]:
print(percentage)

38.663493169565214


Q6. Trong customers.csv, xét các khách hàng có age_group khác null, nhóm tuổi nào có số
đơn hàng trung bình trên mỗi khách hàng cao nhất? (tổng số đơn / số khách hàng trong
nhóm)

In [36]:
customers = pd.read_csv('customers.csv')

In [37]:
customers.head()

,customer_id,zip,city,signup_date,gender,age_group,acquisition_channel
0,1,15201,Hai Phong,2021-12-30,Female,35-44,social_media
1,2,15201,Hai Phong,2013-12-27,Female,45-54,email_campaign
2,3,15201,Hai Phong,2018-07-24,Female,18-24,organic_search
3,4,15201,Hai Phong,2017-11-29,Male,35-44,referral
4,5,15201,Hai Phong,2022-09-23,Male,55+,organic_search


In [38]:
customers.isnull().sum()

customer_id            0
zip                    0
city                   0
signup_date            0
gender                 0
age_group              0
acquisition_channel    0
dtype: int64

In [39]:
q6 = pd.merge( customers, orders, on='customer_id', how='left')

In [40]:
q6.head()

,customer_id,zip_x,city,signup_date,gender,age_group,acquisition_channel,order_id,order_date,zip_y,order_status,payment_method,device_type,order_source
0,1,15201,Hai Phong,2021-12-30,Female,35-44,social_media,5280.0,2012-07-25,15201.0,delivered,cod,desktop,paid_search
1,1,15201,Hai Phong,2021-12-30,Female,35-44,social_media,184922.0,2014-05-31,15201.0,returned,credit_card,mobile,referral
2,1,15201,Hai Phong,2021-12-30,Female,35-44,social_media,308113.0,2015-07-31,15201.0,delivered,cod,mobile,paid_search
3,1,15201,Hai Phong,2021-12-30,Female,35-44,social_media,483190.0,2017-04-23,15201.0,delivered,cod,mobile,paid_search
4,1,15201,Hai Phong,2021-12-30,Female,35-44,social_media,702081.0,2020-02-24,15201.0,delivered,credit_card,mobile,organic_search


In [41]:
count_order=(q6.groupby('age_group')['order_id'].count())

In [42]:
count_order 

age_group
18-24     89057
25-34    190622
35-44    170368
45-54    124138
55+       72760
Name: order_id, dtype: int64

In [43]:
count_people=q6.groupby('age_group')['customer_id'].nunique()


In [44]:
count_people 

age_group
18-24    17039
25-34    36342
35-44    31920
45-54    23172
55+      13457
Name: customer_id, dtype: int64

In [45]:
percentage_1_cus = (count_order/count_people)

In [46]:
percentage_1_cus

age_group
18-24    5.226656
25-34    5.245226
35-44    5.337343
45-54    5.357241
55+      5.406851
dtype: float64

In [47]:
import pandas as pd

orders = pd.read_csv("orders.csv")
customers = pd.read_csv("customers.csv")

# chỉ lấy khách có age_group không null
cust_age = customers[customers["age_group"].notna()][["customer_id", "age_group"]]

# tổng đơn theo khách
orders_per_cust = orders.groupby("customer_id")["order_id"].count().rename("n_orders")

# gộp và tính trung bình số đơn/khách theo nhóm tuổi
tmp = (cust_age.merge(orders_per_cust, on="customer_id", how="left")
              .assign(n_orders=lambda d: d["n_orders"].fillna(0))
              .groupby("age_group")["n_orders"]
              .mean()
              .sort_values(ascending=False))

print("Nhóm tuổi có số đơn TB/khách cao nhất:", tmp.index[0])
print("Số đơn TB/khách:", tmp.iloc[0])

tmp

Nhóm tuổi có số đơn TB/khách cao nhất: 55+
Số đơn TB/khách: 5.406851452775507


age_group
55+      5.406851
45-54    5.357241
35-44    5.337343
25-34    5.245226
18-24    5.226656
Name: n_orders, dtype: float64

Q7. Vùng (region) nào trong geography.csv tạo ra tổng doanh thu cao nhất trong
sales_train.csv?

In [48]:
geography = pd.read_csv('geography.csv')

In [49]:
geography.head(3)

,zip,city,region,district
0,15201,Hai Phong,East,District #13
1,15202,Phu Ly,East,District #13
2,15203,Viet Tri,East,District #13


In [50]:
sales_train = pd.read_csv('sales.csv')
sales_train.head()

,Date,Revenue,COGS
0,2012-07-04,5123547.94,3982991.19
1,2012-07-05,2751773.45,2150580.23
2,2012-07-06,3054029.42,2517632.84
3,2012-07-07,2667930.94,2108246.62
4,2012-07-08,2360851.90,1808622.79


In [51]:
sales_train.shape

(3833, 3)

In [52]:
orders = pd.read_csv('orders.csv')
orders.head()

,order_id,order_date,customer_id,zip,order_status,payment_method,device_type,order_source
0,1,2012-07-04,58578,1109,delivered,credit_card,desktop,paid_search
1,2,2012-07-04,58621,1330,returned,cod,mobile,paid_search
2,3,2012-07-04,58811,1473,delivered,credit_card,desktop,direct
3,4,2012-07-04,59453,2360,delivered,credit_card,desktop,referral
4,6,2012-07-06,57821,2886,delivered,paypal,mobile,email_campaign


In [53]:
orders['order_date'].nunique()

3833

In [54]:
sales_train = sales_train.rename(columns= {"Date": "order_date"})

In [55]:
new_orders = pd.merge(orders, sales_train, on = 'order_date', how = 'left')

In [56]:
new_orders = pd.merge(new_orders, geography, on = 'zip', how = 'left')

In [57]:
new_orders.head()

,order_id,order_date,customer_id,zip,order_status,payment_method,device_type,order_source,Revenue,COGS,city,region,district
0,1,2012-07-04,58578,1109,delivered,credit_card,desktop,paid_search,5123547.94,3982991.19,Hanoi,East,District #02
1,2,2012-07-04,58621,1330,returned,cod,mobile,paid_search,5123547.94,3982991.19,Phu Ly,East,District #02
2,3,2012-07-04,58811,1473,delivered,credit_card,desktop,direct,5123547.94,3982991.19,Lao Cai,East,District #02
3,4,2012-07-04,59453,2360,delivered,credit_card,desktop,referral,5123547.94,3982991.19,Son Tay,East,District #02
4,6,2012-07-06,57821,2886,delivered,paypal,mobile,email_campaign,3054029.42,2517632.84,Uong Bi,East,District #02


In [58]:
payments = pd.read_csv('payments.csv')

In [59]:
new_orders = pd.merge(new_orders, payments, on = 'order_id', how='left')

In [60]:
print(new_orders.groupby('region')['payment_value'].sum().idxmax())

East


Q8. Trong các đơn hàng có order_status = ’cancelled’ trong orders.csv, phương thức
thanh toán nào được sử dụng nhiều nhất?

In [61]:
orders = pd.read_csv('orders.csv')
orders.head()

,order_id,order_date,customer_id,zip,order_status,payment_method,device_type,order_source
0,1,2012-07-04,58578,1109,delivered,credit_card,desktop,paid_search
1,2,2012-07-04,58621,1330,returned,cod,mobile,paid_search
2,3,2012-07-04,58811,1473,delivered,credit_card,desktop,direct
3,4,2012-07-04,59453,2360,delivered,credit_card,desktop,referral
4,6,2012-07-06,57821,2886,delivered,paypal,mobile,email_campaign


In [62]:
q8 = orders[orders['order_status'] == 'cancelled']

In [63]:
q8['payment_method'].value_counts().idxmax()

'credit_card'

In [64]:
import pandas as pd

orders = pd.read_csv("orders.csv")

q8 = orders[orders["order_status"] == "cancelled"]

most_method = q8["payment_method"].value_counts().idxmax()
most_count  = q8["payment_method"].value_counts().max()

print("Phương thức thanh toán dùng nhiều nhất:", most_method)
print("Số đơn:", most_count)

# (tuỳ chọn) xem full bảng
q8["payment_method"].value_counts()

Phương thức thanh toán dùng nhiều nhất: credit_card
Số đơn: 28452


payment_method
credit_card      28452
cod              15468
paypal            7817
apple_pay         5190
bank_transfer     2535
Name: count, dtype: int64

Q9. Trong bốn kích thước sản phẩm (S, M, L, XL), kích thước nào có tỷ lệ trả hàng cao
nhất, được định nghĩa là số bản ghi trong returns chia cho số dòng trong order_items (join
với products theo product_id)?

In [65]:
items = pd.read_csv('order_items.csv')
items.head(3)

C:\Users\Admin\AppData\Local\Temp\ipykernel_28380\2154089599.py:1: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  items = pd.read_csv('order_items.csv')


,order_id,product_id,quantity,unit_price,discount_amount,promo_id,promo_id_2
0,1,2400,7,1138.22,0.0,NaN,NaN
1,2,609,7,10166.25,0.0,NaN,NaN
2,3,396,3,11220.33,0.0,NaN,NaN


In [66]:
products = pd.read_csv('products.csv')
products.head(3)

,product_id,product_name,category,segment,size,color,price,cogs
0,536,SaigonFlex UC-01,Streetwear,Everyday,S,green,11059.650000,9704.842875
1,537,SaigonFlex UC-02,Streetwear,Everyday,M,silver,9523.076013,5393.870254
2,538,SaigonFlex UC-03,Streetwear,Everyday,L,pink,15951.633158,11371.919278


In [67]:
new_products = pd.merge(items, products, on = 'product_id', how ='left')

In [68]:
new_products

,order_id,product_id,quantity,unit_price,discount_amount,promo_id,promo_id_2,product_name,category,segment,size,color,price,cogs
0,1,2400,7,1138.22,0.0,NaN,NaN,VietMotion YY-09,GenZ,Trendy,S,red,1109.261061,1053.798008
1,2,609,7,10166.25,0.0,NaN,NaN,SaigonFlex UC-74,Streetwear,Everyday,M,yellow,10426.571034,8987.704231
2,3,396,3,11220.33,0.0,NaN,NaN,SaigonFlex UM-01,Streetwear,Balanced,S,green,11028.428695,10091.012256
3,4,635,5,10639.25,0.0,NaN,NaN,SaigonFlex UC-00,Streetwear,Everyday,XL,purple,10745.220588,9205.430478
4,6,1935,1,1597.84,0.0,NaN,NaN,UrbanVN RP-10,Outdoor,Activewear,XL,purple,1609.911509,1048.696357
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
714664,834372,690,8,4473.92,0.0,NaN,NaN,SaigonFlex UC-55,Streetwear,Everyday,L,red,4598.817231,3100.062695
714665,834377,1995,7,5250.79,0.0,NaN,NaN,UrbanVN UM-02,Streetwear,Balanced,XL,purple,5159.312851,3025.421056
714666,834387,2331,8,7389.06,0.0,NaN,NaN,VietMotion UE-05,Streetwear,Performance,XL,black,7365.661770,6671.816431
714667,834392,1115,5,4767.33,0.0,NaN,NaN,MekongFit RS-03,Outdoor,Premium,XL,purple,4828.925117,4587.478861


In [69]:
returns = pd.read_csv('returns.csv')
returns.head()

,return_id,order_id,product_id,return_date,return_reason,return_quantity,refund_amount
0,RET-000001,2,609,2012-07-25,late_delivery,6,52458.01
1,RET-000002,32,1862,2012-07-16,wrong_size,2,5141.37
2,RET-000003,35,2359,2012-07-16,wrong_size,1,5315.95
3,RET-000004,47,1449,2012-07-11,wrong_size,4,6493.75
4,RET-000005,47,1450,2012-07-25,wrong_size,1,1740.76


In [70]:
returns = pd.merge(returns, products, on = 'product_id', how ='left')
returns.head()

,return_id,order_id,product_id,return_date,return_reason,return_quantity,refund_amount,product_name,category,segment,size,color,price,cogs
0,RET-000001,2,609,2012-07-25,late_delivery,6,52458.01,SaigonFlex UC-74,Streetwear,Everyday,M,yellow,10426.571034,8987.704231
1,RET-000002,32,1862,2012-07-16,wrong_size,2,5141.37,SaigonCore YY-57,GenZ,Trendy,L,orange,2656.232069,1842.628186
2,RET-000003,35,2359,2012-07-16,wrong_size,1,5315.95,VietMotion UC-07,Streetwear,Everyday,XL,yellow,5399.825901,3136.758866
3,RET-000004,47,1449,2012-07-11,wrong_size,4,6493.75,VietMode RP-41,Outdoor,Activewear,M,yellow,1802.115000,1575.769356
4,RET-000005,47,1450,2012-07-25,wrong_size,1,1740.76,VietMode RP-42,Outdoor,Activewear,L,red,1802.115000,1309.056336


In [71]:
num_size = new_products['size'].value_counts()

In [72]:
num_size

size
XL    193025
M     176428
L     173174
S     172042
Name: count, dtype: int64

In [73]:
return_size = returns['size'].value_counts()

In [74]:
((return_size/num_size)*100).idxmax()

'S'

In [75]:
import pandas as pd

# 1. Đọc dữ liệu
products = pd.read_csv('products.csv')
returns = pd.read_csv('returns.csv')
order_items = pd.read_csv('order_items.csv', low_memory=False)

# 2. Join cả 2 bảng với products để lấy được cột 'size'
ret_size = returns.merge(products[['product_id', 'size']], on='product_id', how='inner')
oi_size  = order_items.merge(products[['product_id', 'size']], on='product_id', how='inner')

# 3. Đếm số dòng của từng size
tu_so  = ret_size['size'].value_counts()
mau_so = oi_size['size'].value_counts()

# 4. Tính tỷ lệ (chia 2 Series có chung index S, M, L, XL)
ty_le = (tu_so / mau_so) * 100

print(ty_le.loc[['S', 'M', 'L', 'XL']].sort_values(ascending=False))

size
S     5.651527
L     5.624978
M     5.566010
XL    5.520010
Name: count, dtype: float64


Q10. Trong payments.csv, kế hoạch trả góp nào có giá trị thanh toán trung bình trên
mỗi đơn hàng cao nhất?
A) 1 kỳ (trả một lần)
B) 3 kỳ
C) 6 kỳ
D) 12 kỳ

In [76]:
payments = pd.read_csv('payments.csv')
payments.head()

,order_id,payment_method,payment_value,installments
0,1,credit_card,7967.54,3
1,2,cod,71163.75,1
2,3,credit_card,33660.99,3
3,4,credit_card,53196.25,3
4,6,paypal,1597.84,1


In [77]:
payments['installments'].value_counts()

installments
1     262866
3     218949
6     109910
12     54126
2       1094
Name: count, dtype: int64

In [78]:
mean_payment = payments.groupby('installments')['payment_value'].mean()

In [80]:
mean_payment.idxmax()

np.int64(6)